In [11]:
from datetime import datetime, timezone
from pathlib import Path
import json
from aind_data_schema_models.organizations import Organization, OrganizationModel
from aind_data_schema_models.harp_types import HarpDeviceType, HarpDeviceTypeModel
from aind_data_schema_models.modalities import Modality
from aind_data_schema_models.slap2_acquisition_type import Slap2AcquisitionType
from aind_data_schema_models.units import PowerUnit, SizeUnit, FrequencyUnit, TimeUnit
from aind_data_schema.components.identifiers import Code
from aind_data_schema.components.devices import Filter
from aind_data_schema.components.coordinates import Translation, Scale, CoordinateSystemLibrary, Origin
from aind_data_schema.core.instrument import Instrument
from aind_data_schema.core.acquisition import (
    Acquisition,
    DataStream,
    StimulusEpoch,
)
from aind_data_schema.components.configs import (
    Channel,
    DetectorConfig,
    LaserConfig,
    TriggerType,
    ImagingConfig,
    Slap2Plane,
    PlanarImage,
    DeviceConfig,
)
from aind_data_schema_models.brain_atlas import CCFv3

from aind_data_schema_models.stimulus_modality import StimulusModality

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [12]:
origin_ari = CoordinateSystemLibrary.BREGMA_ARI
origin_ari.name = "Arbitrary Origin ARI"
origin_ari.axis_unit = SizeUnit.UM
origin_ari.origin = Origin.ORIGIN

instrument_json_path = r"\\allen\aind\scratch\ophys\Andrew\metadata\instrument.json"
instrument_json = Path(instrument_json_path).read_text()
instrument = Instrument.model_validate_json(instrument_json)
instrument_components = instrument.get_component_names()

In [14]:
# temporary fix to remove manufacturer and harp type names from instrument components (recent updates in main branch fix this)
orgs = Organization()
manufacturer_names = {getattr(orgs, attr).name for attr in dir(orgs) if isinstance(getattr(orgs, attr), OrganizationModel)}

harp_types = HarpDeviceType()
harp_type_names = {getattr(harp_types, attr).name for attr in dir(harp_types) if isinstance(getattr(harp_types, attr), HarpDeviceTypeModel)}

instrument_components = [name for name in instrument_components if name not in manufacturer_names and name not in harp_type_names]
# If a timezone isn't specified, the timezone of the computer running this
# script will be used as default

In [15]:
##REPLACE: actual start time extracted from files
harp_start_time = datetime(2022, 7, 12, 7, 00, 00, tzinfo=timezone.utc)
harp_end_time = datetime(2022, 7, 12, 7, 30, 00, tzinfo=timezone.utc)

face_camera_exposure = 10
body_camera_exposure = 10
eye_camera_exposure = 10

In [16]:
project_name = "SLAP2"
acquisition_type = "multi-roi raster drifting gratings"

num_paths = 2
active_channels = ["Red", "Green"]

plane_depths = {"Path 1": [184, 160], "Path 2": [100, 105]}
hwp_laser_power = 75 # in percent

x_dilations = {"Path 1": 9, "Path 2": 9}

channel_intended_measurements = {
    "Red": "RCaMP3",
    "Green": "iGluSnFR4s",
}

In [17]:
stage_offset_from_origin = None # or [0, 0] in um

imaging_target_name = "Neuron1"

In [20]:
slap2_plane_rois_raster = {"Path 1":
    [
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 1"][0], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.RASTER,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 1"],
            dilation_unit=SizeUnit.PX,
        ),
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 1"][1], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.RASTER,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 1"],
            dilation_unit=SizeUnit.PX,
        ),
    ],
    "Path 2": [
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 2"][0], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.RASTER,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 2"],
            dilation_unit=SizeUnit.PX,
        ),
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 2"][1], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.RASTER,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 2"],
            dilation_unit=SizeUnit.PX,
        ),
    ]
}

In [21]:
slap2_plane_rois_integration = {"Path 1": 
    [
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 1"][0], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.INTEGRATION,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 1"],
            dilation_unit=SizeUnit.PX,
        ),
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 1"][1], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.INTEGRATION,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 1"],
            dilation_unit=SizeUnit.PX,
        ),
    ],
    "Path 2": [
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 2"][0], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.INTEGRATION,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 2"],
            dilation_unit=SizeUnit.PX,
        ),
        Slap2Plane( # each fastZ plane will get its own SLap2Plane
            depth=plane_depths["Path 2"][1], # get depth from SLAP2 metadata and user-inputted pia remote focus depth
            depth_unit=SizeUnit.UM,
            power=hwp_laser_power, # get power percent from SLAP2 metadata
            power_unit=PowerUnit.PERCENT,
            targeted_structure=CCFv3.VISPL2_3,
            # SLAP2 specific metadata
            target_name=imaging_target_name, # get targeted neurons from user input
            slap2_acquisition_type=Slap2AcquisitionType.INTEGRATION,
            superpixel_image_path="path/to/mask.tif", # get mask from SLAP2 metadata
            unique_frame_rates=[], # get list of unique frame rates used from SLAP2 metadata
            frame_rate_unit=FrequencyUnit.HZ,
            frame_rate_image_path="path/to/frame_rate_image.tif", # get frame rate image from SLAP2 metadata
            unique_y_dilations=[], # get list of unique dilations used from SLAP2 metadata
            y_dilation_image_path="path/to/dilation_image.tif", # get dilation image from SLAP2 metadata
            x_dilation=x_dilations["Path 2"],
            dilation_unit=SizeUnit.PX,
        ),
    ],
}

In [22]:
monaco_laser_config = LaserConfig(
    device_name="Monaco 150",
    wavelength=1030,
    wavelength_unit=SizeUnit.NM,
    power=150, # max power of laser
    power_unit=PowerUnit.MW,
)

In [23]:
sipm_config = {
    "Red": DetectorConfig(
        device_name="Red SiPM",
        trigger_type=TriggerType.INTERNAL,
    ),
    "Green": DetectorConfig(
        device_name="Green SiPM",
        trigger_type=TriggerType.INTERNAL,
    ),
}

In [24]:
a = Acquisition(
    subject_id="12345",
    acquisition_start_time=harp_start_time,
    acquisition_end_time=harp_end_time,
    experimenters=["John Smith"],
    ethics_review_id=["2415"],
    instrument_id="SLAP2_1_VCO_1",
    acquisition_type=project_name + ": " + acquisition_type,
    notes="center back of cranial window is coordinate system origin",
    coordinate_system=origin_ari,
    calibrations=[],
    maintenance=[],
    data_streams=[
        DataStream(
            stream_start_time=harp_start_time,
            stream_end_time=harp_end_time,
            modalities=[Modality.SLAP2, Modality.BEHAVIOR_VIDEOS, Modality.BEHAVIOR],
            active_devices=instrument_components,
            configurations=[
                ImagingConfig(
                    device_name="SLAP2_1",
                    channels=[
                        Channel(
                            channel_name=f"Path {path_idx + 1} {channel_color} channel",
                            intended_measurement=channel_intended_measurements[channel_color],
                            detector=sipm_config[channel_color],
                            light_sources=[monaco_laser_config],
                            additional_device_names=[
                                DeviceConfig(device_name=f"DMD{path_idx + 1}"),
                                DeviceConfig(device_name="Leica Objective"),
                                DeviceConfig(device_name="Half-Wave Plate"),
                                DeviceConfig(device_name="Polygonal Scanner"),
                            ],
                            emission_filters=[DeviceConfig(device_name=f"{channel_color} Emission Filters")],
                            emission_wavelength=next((c for c in instrument.components if isinstance(c, Filter) and c.name == f"{channel_color} Emission Filters")).center_wavelength,
                            emission_wavelength_unit=next((c for c in instrument.components if isinstance(c, Filter) and c.name == f"{channel_color} Emission Filters")).wavelength_unit,
                        ) for path_idx in range(num_paths) for channel_color in active_channels
                    ],
                    images=[
                        PlanarImage(
                            channel_name=f"Path {path_idx + 1} {channel_color} channel",
                            # can add location of stage here if we know the relative position to Bregma
                            image_to_acquisition_transform=[
                                Scale(
                                    scale=[0.25, 0.25],
                                ),
                                *([Translation(
                                        translation=stage_offset_from_origin,
                                    )]
                                    if stage_offset_from_origin is not None
                                    else []
                                ),
                            ],
                            dimensions=Scale(
                                scale=[800, 1280],
                            ),
                            dimensions_unit=SizeUnit.PX,
                            planes=slap2_plane_rois_raster[f"Path {path_idx + 1}"]+slap2_plane_rois_integration[f"Path {path_idx + 1}"],
                        )
                        for path_idx in range(num_paths) for channel_color in active_channels
                    ],
                ),
                DetectorConfig(
                    device_name="FaceCamera",
                    exposure_time=face_camera_exposure,
                    exposure_time_unit=TimeUnit.MS,
                    trigger_type=TriggerType.EXTERNAL,
                ),
                DetectorConfig(
                    device_name="BodyCamera",
                    exposure_time=body_camera_exposure,
                    exposure_time_unit=TimeUnit.MS,
                    trigger_type=TriggerType.EXTERNAL,
                ),
                DetectorConfig(
                    device_name="EyeCamera",
                    exposure_time=eye_camera_exposure,
                    exposure_time_unit=TimeUnit.MS,
                    trigger_type=TriggerType.EXTERNAL,
                ),
            ],
        )
    ],
    stimulus_epochs=[
        StimulusEpoch(
            stimulus_start_time=harp_start_time,
            stimulus_end_time=harp_end_time,
            stimulus_name="Shuffled 8-direction drifting gratings",
            code=Code(
                url="https://github.com/AllenNeuralDynamics/ophys-passive-visual-stim/blob/main/src/RandomDriftingGratings_ContinuousTrials.bonsai",
                version="168e4ef1923c535f6c4d914a126526cc11168ac7", # get git commit id
                name="RandomDriftingGratings_ContinuousTrials.bonsai",
                language="Bonsai",
                language_version="2.9.0", # get from repo or code directly
                parameters={
                    "Initial Spont Int": 28,
                    "Num Trials": 40,
                    "PortName": "COM7",
                    "Screen_BlueColor (0-1)": 1,
                    "Screen_GreenColor (0-1)": 0,
                    "Screen_RedColor (0-1)": 0,
                    "Subject": "000000",
                    "GratingTrialParametersFile": "ParameterFiles/8_direction_drifting_grating_params.csv"
                }
            ),
            stimulus_modalities=[StimulusModality.VISUAL],
            notes="", # include description of stimulus here if code and/or parameters are not available
            active_devices=["Stimulus Screen"],
            configurations=[],
        ),
    ],
)

In [25]:
if __name__ == "__main__":
    serialized = a.model_dump_json()
    deserialized = Acquisition.model_validate_json(serialized)
    deserialized.write_standard_file(prefix="slap2")